# Symulacje numeryczne
 Schematy Rungego-Kutty-Nyströma dla równań różniczkowych zwyczajnych z rozwiązaniem okresowym  
Wioletta Małgorzata Sudoł  

Niniejszy notatnik zawiera implementację oraz analizę numeryczną algorytmów badanych w rozdziale 3 pracy magisterskiej.



## 1. Oscylator harmoniczny
* **Równanie testowe:** Równanie postaci $y'' = -\omega^2 y$ z parametrem $\omega = 1$, warunkami początkowymi $y(0)=1, y'(0)=0$ oraz z rozwiązaniem analitycznym $y(t) = \cos(t)$; rozważany przedział $[0, 40\pi]$ oraz krok początkowy $h = 2\pi$ zmniejszany dwukrotnie w pętli.
* **Miara błędu:** Błąd końcowy ($error\_end$).
* **Analiza wrażliwości:** W tej samej sekcji kodu przeprowadzane jest badanie stabilności metody $trig1$ przy użyciu zmiennej `omega_met`.

In [1]:
import numpy as np
import pandas as pd

In [2]:
# parametry początkowe
omega = 1  # rzeczywista częstotliwość układu
omega_met = 1 # częstotliwość metody - do analizy wrażliwości, w eksperymentach testowano wartości zaburzeń: ±5% (1.05 i 0.95), ±10% (1.10 i 0.90) oraz ±20% (1.20 i 0.80)
y0 = 1 # y(0)=1
yp0 = 0 # y'(0)=0

# przedział [0,40*pi]
t0 = 0
tN = 40 * np.pi

def rozw(t): # rozwiązanie analityczne
    return np.cos(omega * t)

In [3]:
# trig1 — współczynniki z artykułu

c1 = 1 / 4
c2 = 3 / 4


def P(nu):
    return np.cos(c1 * nu) * np.sin(c2 * nu) - np.sin(c1 * nu) * np.cos(c2 * nu)


def a11(nu):
    return (
        ((1 - np.cos(c1 * nu)) / nu**2 * np.sin(c2 * nu)
         - np.cos(c2 * nu) * (c1 * nu - np.sin(c1 * nu)) / nu**2)
        / P(nu)
    )

def a12(nu):
    return (
        (np.cos(c1 * nu) * (c1 * nu - np.sin(c1 * nu)) / nu**2
         - np.sin(c1 * nu) * (1 - np.cos(c1 * nu)) / nu**2)
        / P(nu)
    )

def a21(nu):
    return (
        ((1 - np.cos(c2 * nu)) / nu**2 * np.sin(c2 * nu)
         - np.cos(c2 * nu) * (c2 * nu - np.sin(c2 * nu)) / nu**2)
        / P(nu)
    )

def a22(nu):
    return (
        (np.cos(c1 * nu) * (c2 * nu - np.sin(c2 * nu)) / nu**2
         - np.sin(c1 * nu) * (1 - np.cos(c2 * nu)) / nu**2)
        / P(nu)
    )

def b1(nu):
    return (
        ((1 - np.cos(nu)) / nu**2 * np.sin(c2 * nu)
         - np.cos(c2 * nu) * (nu - np.sin(nu)) / nu**2)
        / P(nu)
    )

def b2(nu):
    return (
        (np.cos(c1 * nu) * (nu - np.sin(nu)) / nu**2
         - np.sin(c1 * nu) * (1 - np.cos(nu)) / nu**2)
        / P(nu)
    )

def bp1(nu):
    return (
        ((np.sin(nu) / nu) * np.sin(c2 * nu)
         - np.cos(c2 * nu) * (1 - np.cos(nu)) / nu)
        / P(nu)
    )

def bp2(nu):
    return (
        (np.cos(c1 * nu) * (1 - np.cos(nu)) / nu
         - np.sin(c1 * nu) * (np.sin(nu) / nu))
        / P(nu)
    )

In [4]:
# dla metody trig1
def RKN_trig1(t0, t_N, h, y0, yp0, omega_met, omega):

    N = int(np.ceil((t_N - t0) / h))  # liczba kroków
    h = (t_N - t0) / N  # długość kroku
    t = np.linspace(t0, t_N, N + 1) # podział odcinka t

    y = np.zeros(N + 1)
    yp = np.zeros(N + 1)
    y[0] = y0   # y(t_0)=y_0
    yp[0] = yp0  # y'(t_0)=y'_0

    nu = omega_met * h

    A = np.array([[a11(nu), a12(nu)],[a21(nu), a22(nu)]])  # macierz A

    b = np.array([b1(nu), b2(nu)]) # wektor b
    bp = np.array([bp1(nu), bp2(nu)])  # wektor b'
    c = np.array([c1, c2]) # wektor c

    # zależy tylko od y, więc f(y)=-omega^2*y
    # przypisuję u=y_n+c_i*h*y'_n
    # dla przykładu y''=-omega^2*y mamy Y=u+h^2*A*f(Y)
    # f(Y)=-omega^2*Y => Y=u-h^2*omega^2*A*Y
    # (I+h^2*omega^2*A)Y=u, (I+h^2*omega^2*A)=M, macierz M - stała

    for n in range(N):
        u = y[n] + c * h * yp[n]
        M = np.eye(2) + h**2 * omega**2 * A
        Y = np.linalg.solve(M, u)  # rozwiązanie M*Y=u
        f = -omega**2 * Y

        # aktualizacja rozw.
        y[n + 1] = y[n] + h * yp[n] + h ** 2 * np.dot(b, f)
        yp[n + 1] = yp[n] + h * np.dot(bp, f)

    return pd.DataFrame({"t": t, "y": y})


In [5]:
# dla pozostałych metod
def RKN_standard(t0, t_N, h, y0, yp0, omega, A, b, bp, c):
    N = int(np.ceil((t_N - t0) / h))
    h = (t_N - t0) / N
    t = np.linspace(t0, t_N, N + 1)
    y, yp = np.zeros(N + 1), np.zeros(N + 1)
    y[0], yp[0] = y0, yp0

    M = np.eye(len(c)) + h ** 2 * omega ** 2 * A

    for n in range(N):
        u = y[n] + c * h * yp[n]
        Y = np.linalg.solve(M, u)
        f = -omega ** 2 * Y
        y[n + 1] = y[n] + h * yp[n] + h ** 2 * np.dot(b, f)
        yp[n + 1] = yp[n] + h * np.dot(bp, f)

    return pd.DataFrame({"t": t, "y": y})

In [6]:
def generate_h(h0, n, mode="half"):
    h_vals = [h0]
    for _ in range(n - 1):
        if mode == "half":
            h_vals.append(h_vals[-1] / 2) # połowienie kroku
        elif mode == "div1.1":
            h_vals.append(h_vals[-1] / 1.1) # dzielenie kroku przez 1.1
    return h_vals

h_values = generate_h(2*np.pi, 10, mode="half") # ustawiamy "half" lub "div1.1"


In [7]:
y_dok = rozw(tN) # wartość referencyjna rozwiązania dokładnego na końcu przedziału

# obliczamy błąd i cd, iloraz oraz rząd numeryczny
def calc_res_1(method_name, A=None, b=None, bp=None, c=None):
    res = pd.DataFrame({
        "h": h_values,
        "error_end": [np.nan] * len(h_values),
        "cd": [np.nan] * len(h_values),
        "iloraz" : [np.nan] * len(h_values),
        "p_num": [np.nan] * len(h_values)
    })

    for i, h in enumerate(h_values):
        if method_name == "trig1":
            df = RKN_trig1(t0, tN, h, y0, yp0, omega_met, omega)
        else:
            df = RKN_standard(t0, tN, h, y0, yp0, omega, A, b, bp, c)

        y_end = df["y"].iloc[-1]
        err_abs = abs(y_end - y_dok)
        cd = -np.log10(err_abs/ abs(y_dok) )
        res.loc[i, "error_end"] = err_abs
        res.loc[i, "cd"] = cd

        if i > 0:
          err_prev = res.loc[i-1, "error_end"]
          iloraz = err_prev / err_abs
          res.loc[i, "iloraz"] = iloraz
          res.loc[i, "p_num"] = np.log2(iloraz)

    return res

In [8]:
# współczynniki metod
# RKN2-4
A_24  = np.array([[1/2,0],[-5/12,1/2]])
b_24  = np.array([0,1/2])
bp_24 = np.array([0,1])
c_24  = np.array([1/2,1/2])

# RKN2-10
A_210 = np.array([
    [0.052320267566927, 0, 0],
    [-0.17329232352333, 0.052320267566927, 0],
    [-0.01271397498318, 0.043727040749588, 0.052320267566927]
])
b_210 = np.array([0, 0, 1/2])
bp_210 = np.array([0, 0, 1])
c_210 = np.array([1/2, 3/10, 1/2])

# RKN4-4
A_44  = np.array([[1/6+1/12*np.sqrt(3),0],[-1/6*np.sqrt(3),1/6+1/12*np.sqrt(3)]])
b_44  = np.array([1/4-1/12*np.sqrt(3),1/4+1/12*np.sqrt(3)])
bp_44 = np.array([1/2,1/2])
c_44  = np.array([1/2+1/6*np.sqrt(3),1/2-1/6*np.sqrt(3)])

### Wyniki numeryczne dla oscylatora harmonicznego

In [9]:
results_trig = calc_res_1("trig1")
results_rkn24 = calc_res_1("RKN24", A_24, b_24, bp_24, c_24)
results_rkn210 = calc_res_1("RKN210", A_210, b_210, bp_210, c_210)
results_rkn44 = calc_res_1("RKN44", A_44, b_44, bp_44, c_44)

print("--- METODA trig1 ---")
print(results_trig)
print("\n--- METODA RKN2-4 ---")
print(results_rkn24)
print("\n--- METODA RKN2-10 ---")
print(results_rkn210)
print("\n--- METODA RKN4-4 ---")
print(results_rkn44)

--- METODA trig1 ---
          h     error_end         cd     iloraz     p_num
0  6.283185  9.325873e-15  14.030310        NaN       NaN
1  3.141593  4.440892e-16  15.352530  21.000000  4.392317
2  1.570796  4.662937e-15  14.331340   0.095238 -3.392317
3  0.785398  1.110223e-16  15.954590  42.000000  5.392317
4  0.392699  1.332268e-15  14.875409   0.083333 -3.584963
5  0.196350  4.773959e-15  14.321121   0.279070 -1.841302
6  0.098175  5.040413e-14  13.297534   0.094714 -3.400284
7  0.049087  3.042011e-14  13.516839   1.656934  0.728516
8  0.024544  1.529887e-13  12.815341   0.198839 -2.330328
9  0.012272  4.042322e-13  12.393369   0.378467 -1.401759

--- METODA RKN2-4 ---
          h     error_end         cd      iloraz     p_num
0  6.283185  1.999931e+00  -0.301015         NaN       NaN
1  3.141593  1.464351e+00  -0.165645    1.365746  0.449689
2  1.570796  1.991390e+00  -0.299156    0.735341 -0.443514
3  0.785398  1.787857e+00  -0.252333    1.113842  0.155545
4  0.392699  2.420365e-

## 2. Oscylator harmoniczny z wymuszeniem
* **Równanie testowe:** Równanie postaci $y'' = -\omega^2 y + (\omega^2 - 1)\sin(t)$ z parametrem $\omega = 10$, warunkami początkowymi $y(0)=1, y'(0)=\omega+1$ oraz rozwiązaniem analitycznym $y(t) = \cos(\omega t) + \sin(\omega t) + \sin(t)$; rozważamy przedział $[0, 20\pi]$ oraz krok zmniejszany w pętli od $h = 2\pi$.
* **Miary błędu:** Testowane są dwie różne miary błędu dla wszystkich czterech schematów: błąd końcowy ($error\_end$) oraz błąd maksymalny ($error\_max$).

In [10]:
# parametry początkowe
omega = 10 # rzeczywista częstotliwość układu
omega_met = 10
y0 = 1 # y(0) = 1
yp0 = omega + 1 # y'(0) = omega + 1

# przedział [0,20*pi]
t0 = 0
tN = 20 * np.pi


def rozw(t, omega): # rozwiązanie analityczne
    return np.cos(omega*t) + np.sin(omega*t) + np.sin(t)


In [11]:
# dla metody trig1

def RKN_trig2(t0, tN, h, y0, yp0, omega_met, omega):
    N = int(np.ceil((tN - t0) / h))  # liczba kroków
    h = (tN - t0) / N   # długość kroku
    t = np.linspace(t0, tN, N + 1)  # podział oscinka t

    y = np.zeros(N + 1)
    yp = np.zeros(N + 1)
    y[0] = y0  # y(t_0)=y_0
    yp[0] = yp0  # y'(t_0)=y'_0

    nu = omega_met * h

    A = np.array([[a11(nu), a12(nu)], # macierz A
                  [a21(nu), a22(nu)]])
    b = np.array([b1(nu), b2(nu)]) # wektor b
    bp = np.array([bp1(nu), bp2(nu)]) # wektor b'
    c = np.array([c1, c2]) # wektor c

    I = np.eye(2)
    LHS = I + (h ** 2) * (omega ** 2) * A

    for n in range(N):
        u = y[n] + c * h * yp[n]
        t_stage = t[n] + c * h
        g = (omega ** 2 - 1) * np.sin(t_stage)

        RHS = u + (h ** 2) * (A @ g)  # operator @ dla mnożenia macierzy

        Y = np.linalg.solve(LHS, RHS) # rozw. LHR*Y=RHS

        f = -omega ** 2 * Y + g

        y[n + 1] = y[n] + h * yp[n] + (h ** 2) * np.sum(b * f)
        yp[n + 1] = yp[n] + h * np.sum(bp * f)

    return pd.DataFrame({"t": t, "y": y})

In [12]:
# dla pozostałych metod
def RKN_standard2(t0, tN, h, y0, yp0, omega, A, b, bp, c):
    N = int(np.ceil((tN - t0) / h))
    h = (tN - t0) / N
    t = np.linspace(t0, tN, N + 1)
    y, yp = np.zeros(N + 1), np.zeros(N + 1)
    y[0], yp[0] = y0, yp0

    LHS = np.eye(len(c)) + (h**2) * (omega**2) * A

    for n in range(N):
        u = y[n] + c * h * yp[n]
        t_stage = t[n] + c * h
        g = (omega ** 2 - 1) * np.sin(t_stage)

        RHS = u + (h ** 2) * (A @ g)  # operator @ dla mnożenia macierzy

        Y = np.linalg.solve(LHS, RHS) # rozw. LHR*Y=RHS

        f = -omega ** 2 * Y + g

        y[n + 1] = y[n] + h * yp[n] + (h ** 2) * np.sum(b * f)
        yp[n + 1] = yp[n] + h * np.sum(bp * f)

    return pd.DataFrame({"t": t, "y": y})

In [13]:
def generate_h(h0, n, mode="half"):

    h_vals = [h0]
    for _ in range(n - 1):
        if mode == "half":
            h_vals.append(h_vals[-1] / 2)
        elif mode == "div1.1":
            h_vals.append(h_vals[-1] / 1.1)
    return h_vals

h_values = generate_h(2*np.pi, 10, mode="half") # ustawiamy "half" lub "div1.1"


In [14]:
y_dok = rozw(tN, omega) # wartość referencyjna rozwiązania dokładnego na końcu przedziału


# obliczamy błąd  i cd, iloraz oraz rząd numeryczny
def calc_end_2(method_name, A=None, b=None, bp=None, c=None):
    res = pd.DataFrame({
        "h": h_values,
        "error_end": [np.nan] * len(h_values), # błąd końcowy
        "cd": [np.nan] * len(h_values),
        "iloraz" : [np.nan] * len(h_values),
        "p_num": [np.nan] * len(h_values)
    })

    for i, h in enumerate(h_values):
        if method_name == "trig1":
            df = RKN_trig2(t0, tN, h, y0, yp0, omega_met, omega)
        else:
            df = RKN_standard2(t0, tN, h, y0, yp0, omega, A, b, bp, c)

        y_end = df["y"].iloc[-1]
        err_abs = abs(y_end - y_dok)
        cd = -np.log10(err_abs/ abs(y_dok) )
        res.loc[i, "error_end"] = err_abs
        res.loc[i, "cd"] = cd

        if i > 0:
          err_prev = res.loc[i-1, "error_end"]
          iloraz = err_prev / err_abs
          res.loc[i, "iloraz"] = iloraz
          res.loc[i, "p_num"] = np.log2(iloraz)

    return res

In [15]:
def calc_max_2(method_name, A=None, b=None, bp=None, c=None):
    res = pd.DataFrame({
        "h": h_values,
        "error_max": [np.nan] * len(h_values), # błąd maksymalny
        "cd": [np.nan] * len(h_values),
        "iloraz" : [np.nan] * len(h_values),
        "p_num": [np.nan] * len(h_values)
    })

    for i, h in enumerate(h_values):
        if method_name == "trig1":
            df = RKN_trig2(t0, tN, h, y0, yp0, omega_met, omega)
        else:
            df = RKN_standard2(t0, tN, h, y0, yp0, omega, A, b, bp, c)

        y_dok_wektor = rozw(df["t"], omega)
        # obliczamy maksymalny błąd na całej trajektorii
        err_max = np.max(np.abs(df["y"] - y_dok_wektor))
        cd = -np.log10(err_max / np.max(np.abs(y_dok_wektor)))
        res.loc[i, "error_max"] = err_max
        res.loc[i, "cd"] = cd

        if i > 0:
          err_prev = res.loc[i-1, "error_max"]
          iloraz = err_prev / err_max
          res.loc[i, "iloraz"] = iloraz
          res.loc[i, "p_num"] = np.log2(iloraz)

    return res

### Wyniki numeryczne dla oscylatora harmonicznego z wymuszeniem

In [16]:
# wyniki dla error_end
print("--- METODA trig1 ---")
print(calc_end_2("trig1"))
print("\n--- METODA RKN2-4 ---")
print(calc_end_2("RKN24", A_24, b_24, bp_24, c_24))
print("\n--- METODA RKN2-10 ---")
print(calc_end_2("RKN210", A_210, b_210, bp_210, c_210))
print("\n--- METODA RKN4-4 ---")
print(calc_end_2("RKN44", A_44, b_44, bp_44, c_44))

--- METODA trig1 ---
          h     error_end         cd        iloraz      p_num
0  6.283185  9.817476e+02  -2.992000           NaN        NaN
1  3.141593  2.503261e-01   0.601494  3.921874e+03  11.937328
2  1.570796  3.397282e-14  13.468868  7.368422e+12  42.744493
3  0.785398  5.484502e-14  13.260863  6.194332e-01  -0.690979
4  0.392699  4.951595e-14  13.305255  1.107623e+00   0.147467
5  0.196350  3.197442e-14  13.495197  1.548611e+00   0.630975
6  0.098175  3.108624e-14  13.507432  1.028571e+00   0.040642
7  0.049087  1.731948e-14  13.761465  1.794872e+00   0.843881
8  0.024544  1.509903e-14  13.821051  1.147059e+00   0.197939
9  0.012272  1.371125e-13  12.862923  1.101215e-01  -3.182832

--- METODA RKN2-4 ---
          h  error_end        cd     iloraz     p_num
0  6.283185   4.014540 -0.603636        NaN       NaN
1  3.141593   5.012946 -0.700093   0.800835 -0.320424
2  1.570796   0.433030  0.363482  11.576449  3.533121
3  0.785398   2.027549 -0.306971   0.213573 -2.227199
4  0

In [17]:
# wyniki dla error_max
print("--- METODA trig1 ---")
print(calc_max_2("trig1"))
print("\n--- METODA RKN2-4 ---")
print(calc_max_2("RKN24", A_24, b_24, bp_24, c_24))
print("\n--- METODA RKN2-10 ---")
print(calc_max_2("RKN210", A_210, b_210, bp_210, c_210))
print("\n--- METODA RKN4-4 ---")
print(calc_max_2("RKN44", A_44, b_44, bp_44, c_44))

--- METODA trig1 ---
          h   error_max        cd       iloraz      p_num
0  6.283185  981.747562 -2.992000          NaN        NaN
1  3.141593    0.426676  0.369901  2300.918617  11.167994
2  1.570796    0.293497  0.833426     1.453766   0.539795
3  0.785398    0.129091  1.190133     2.273563   1.184955
4  0.392699    0.003938  2.773610    32.782763   5.034866
5  0.196350    0.000199  4.069266    19.754020   4.304074
6  0.098175    0.000094  4.401920     2.111219   1.078076
7  0.049087    0.000026  4.954277     3.567440   1.834889
8  0.024544    0.000007  5.550882     3.904871   1.965275
9  0.012272    0.000002  6.150377     3.976447   1.991480

--- METODA RKN2-4 ---
          h  error_max        cd     iloraz     p_num
0  6.283185  10.298447 -1.012772        NaN       NaN
1  3.141593   9.415874 -0.973861   1.093732  0.129260
2  1.570796   4.018358 -0.303019   2.343214  1.228489
3  0.785398   2.855430 -0.154642   1.407269  0.492898
4  0.392699   2.888473 -0.091806   0.988561 -0.0

## 3. Problem Keplera
* **Równanie testowe:** Układ równań postaci $\begin{cases} q_1''(t) = -\frac{q_1}{(q_1^2 + q_2^2)^{3/2}} \\ q_2''(t) = -\frac{q_2}{(q_1^2 + q_2^2)^{3/2}} \end{cases}$ z warunkami początkowymi $q_1(0) = 1 - e, \quad q_2(0) = 0$,  $q_1'(0) = 0, \quad q_2'(0) = \sqrt{\frac{1+e}{1-e}}$, gdzie dla $e=0$ rozwiązaniem analitycznym jest układ $\begin{cases} q_1(t) = \cos(t) \\ q_2(t) = \sin(t) \end{cases}$; rozważamy przedział $[0, 2\pi]$ oraz krok $h$ od $\pi/10$ do $\pi/640$.
* **Miara błędu:** Błąd maksymalny ($error\_max$).

In [18]:
# parametry początkowe
e = 0.0
omega = 1.0 # częstotliwość układu

# przedział [0,2*pi]
t0 = 0
tN = 2 * np.pi

# warunki początkowe
q0 = np.array([1.0 - e, 0.0])
qp0 = np.array([0.0, np.sqrt((1 + e) / (1 - e))])

def f_kepler(q): # siła grawitacyjna
    r = np.sqrt(q[0] ** 2 + q[1] ** 2)
    return -q / r ** 3


In [19]:
# dla metody trig1
def RKN_trig1_kepler(t0, tN, h, q0, qp0, omega, max_iter=10):
    N = int(np.ceil((tN - t0) / h))  # liczba kroków
    h = (tN - t0) / N  # długość kroku
    t = np.linspace(t0, tN, N + 1) # podział odcinka t
    q = np.zeros((N + 1, 2));
    qp = np.zeros((N + 1, 2))
    q[0], qp[0] = q0, qp0

    nu = omega * h
    A = np.array([[a11(nu), a12(nu)], [a21(nu), a22(nu)]])
    b = np.array([b1(nu), b2(nu)]);
    bp = np.array([bp1(nu), bp2(nu)])
    c = np.array([c1, c2])

    # metoda iteracji prostych
    for n in range(N):
        Y1 = q[n] + c[0] * h * qp[n]
        Y2 = q[n] + c[1] * h * qp[n]

        for _ in range(max_iter):
            Y1_old, Y2_old = Y1.copy(), Y2.copy()

            f1 = f_kepler(Y1)
            f2 = f_kepler(Y2)

            Y1 = q[n] + c[0] * h * qp[n] + (h ** 2) * (A[0, 0] * f1 + A[0, 1] * f2)
            Y2 = q[n] + c[1] * h * qp[n] + (h ** 2) * (A[1, 0] * f1 + A[1, 1] * f2)

        f1, f2 = f_kepler(Y1), f_kepler(Y2)
        q[n + 1] = q[n] + h * qp[n] + (h ** 2) * (b[0] * f1 + b[1] * f2)
        qp[n + 1] = qp[n] + h * (bp[0] * f1 + bp[1] * f2)

    return t, q


In [20]:
# dla pozostałych metod
def RKN_standard_kepler(t0, tN, h, q0, qp0, A, b, bp, c, max_iter=10):
    N = int(np.ceil((tN - t0) / h))  # liczba kroków
    h = (tN - t0) / N  # długość kroku
    t = np.linspace(t0, tN, N + 1) # podział odcinka t
    q = np.zeros((N + 1, 2));
    qp = np.zeros((N + 1, 2))
    q[0], qp[0] = q0, qp0

    s=len(c)

    # metoda iteracji prostych
    for n in range(N):
        Y = q[n] + h * np.outer(c, qp[n])

        for _ in range(max_iter):
            F = np.array([f_kepler(Yi) for Yi in Y])
            Y = q[n] + h * np.outer(c, qp[n]) + (h ** 2) * (A @ F)


        lista_F=[]
        for Yi in Y:
            wynik_F=f_kepler(Yi)
            lista_F.append(wynik_F)
        F=np.array(lista_F)
        q[n + 1] = q[n] + h * qp[n] + (h ** 2) * (b @ F)
        qp[n + 1] = qp[n] + h * (bp @ F)


    return t, q


In [21]:
# obliczamy błąd  i cd, iloraz oraz rząd numeryczny
def calc_1_kepler(method_name, A=None, b=None, bp=None, c=None):
    h_values = [np.pi / 10, np.pi / 20, np.pi / 40, np.pi / 80, np.pi / 160, np.pi / 320, np.pi / 640]

    res = pd.DataFrame({
        "h": h_values,
        "error_max": [np.nan] * len(h_values),
        "cd": [np.nan] * len(h_values),
        "iloraz": [np.nan] * len(h_values),
        "p_num": [np.nan] * len(h_values)
    })

    for i, h in enumerate(h_values):
        if method_name == "trig1":
            t_num, q_num = RKN_trig1_kepler(t0, tN, h, q0, qp0, omega)
        else:
            t_num, q_num = RKN_standard_kepler(t0, tN, h, q0, qp0, A, b, bp, c)

        q_exact = np.array([np.cos(t_num), np.sin(t_num)]).T
        err_max = np.max(np.sqrt(np.sum((q_num - q_exact) ** 2, axis=1)))
        cd = -np.log10(err_max / np.max(np.sqrt(np.sum(q_exact ** 2, axis=1))))
        # zapisywanie wyników do ramki danych przez .loc
        res.loc[i, "error_max"] = err_max
        res.loc[i, "cd"] = cd

        # obliczanie ilorazu i rzędu zbieżności za pomocą logarytmu
        if i > 0:
            err_prev = res.loc[i-1, "error_max"]
            iloraz = err_prev / err_max
            res.loc[i, "iloraz"] = iloraz
            res.loc[i, "p_num"] = np.log2(iloraz)  # wyznaczenie rzędu logarytmem

    return res

### Wyniki numeryczne dla problemu Keplera

In [22]:

print("--- METODA trig1 (KEPLER) ---")
print(calc_1_kepler("trig1"))

print("\n--- METODA RKN2-4 (KEPLER) ---")
print(calc_1_kepler("RKN24", A_24, b_24, bp_24, c_24))

print("\n--- METODA RKN2-10 (KEPLER) ---")
print(calc_1_kepler("RKN210", A_210, b_210, bp_210, c_210))

print("\n--- METODA RKN4-4 (KEPLER) ---")
print(calc_1_kepler("RKN44", A_44, b_44, bp_44, c_44))

--- METODA trig1 (KEPLER) ---
          h     error_max         cd    iloraz     p_num
0  0.314159  1.884371e-14  13.724834       NaN       NaN
1  0.157080  2.265387e-14  13.644858  0.831810 -0.265675
2  0.078540  1.396644e-14  13.854914  1.622022  0.697793
3  0.039270  1.181812e-13  12.927452  0.118178 -3.080964
4  0.019635  2.313100e-13  12.635806  0.510921 -0.968828
5  0.009817  1.211144e-13  12.916804  1.909848  0.933458
6  0.004909  5.548810e-13  12.255800  0.218271 -2.195809

--- METODA RKN2-4 (KEPLER) ---
          h  error_max        cd     iloraz     p_num
0  0.314159   0.105094  0.978423        NaN       NaN
1  0.157080   0.005752  2.240165  18.270146  4.191416
2  0.078540   0.000619  3.208150   9.289346  3.215577
3  0.039270   0.000150  3.824466   4.133480  2.047357
4  0.019635   0.000037  4.429658   4.028949  2.010403
5  0.009817   0.000009  5.032464   4.006879  2.002479
6  0.004909   0.000002  5.634710   4.001715  2.000619

--- METODA RKN2-10 (KEPLER) ---
          h  erro